In [75]:
from dotenv import load_dotenv
import os 
from tavily import TavilyClient
from langchain.chat_models import init_chat_model
from sqlalchemy import create_engine, text
from langchain.tools import tool
load_dotenv()

True

In [76]:
GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# Template Agent

In [77]:
# Fast + Free
cheap_model = init_chat_model(
    "google_genai:gemini-2.5-flash"
)

# Research + General Tasks
middle_model = init_chat_model(
    "google_genai:gemini-2.5-flash"
)

# Advanced Reasoning
advance_model = init_chat_model(
    "groq:llama-3.3-70b-versatile"
).with_fallbacks([middle_model, cheap_model])

In [78]:
from sqlalchemy import create_engine, text
from langchain_core.tools import tool

engine = create_engine(DATABASE_URL)


@tool
def get_user_context(email: str):
    """
    Fetch complete user context including profile, resume, assets,
    templates and AI memory using user email.
    """

    query = text("""
    SELECT

        -- User Profile
        u.id,
        u.full_name,
        u.email,
        u.phone,
        u.role,

        -- Resume Data
        r.raw_text AS resume_text,
        r.skills,
        r.projects,
        r.experience,
        r.education,
        r.social_media_links,
        r.certifications,
        r.objective,

        -- Assets
        (
            SELECT json_agg(
                json_build_object(
                    'name', a.name,
                    'type', a.asset_type,
                    'description', a.description,
                    'content', a.content,
                    'metadata', a.metadata
                )
            )
            FROM assets a
            WHERE a.user_id = u.id
        ) AS assets,


        -- Previous AI Memory
        (
            SELECT json_agg(
                json_build_object(
                    'role', m.role,
                    'message', m.content,
                    'importance', m.importance_score
                )
            )
            FROM ai_memory m
            WHERE m.user_id = u.id
        ) AS memories,


        -- Existing Templates
        (
            SELECT json_agg(
                json_build_object(
                    'name', t.name,
                    'subject', t.subject_line,
                    'text', t.text_content,
                    'html', t.html_content
                )
            )
            FROM templates t
            WHERE t.user_id = u.id
        ) AS templates


    FROM users u

    LEFT JOIN user_resumes r
        ON r.user_id = u.id

    WHERE u.email = :email;

    """)


    with engine.connect() as conn:

        result = conn.execute(
            query,
            {
                "email": email
            }
        )

        row = result.fetchone()


        if not row:
            return {
                "error": "User not found"
            }


        data = dict(row._mapping)


        return {

            "user": {
                "id": str(data["id"]),
                "name": data["full_name"],
                "email": data["email"],
                "phone": data["phone"],
                "role": data["role"]
            },


            "resume": {
                "raw_text": data["resume_text"],
                "skills": data["skills"],
                "projects": data["projects"],
                "experience": data["experience"],
                "education": data["education"],
                "certifications": data["certifications"],
                "objective": data["objective"],
                "social_links": data["social_media_links"]
            },


            "assets": data["assets"] or [],

            "memories": data["memories"] or [],

            "previous_templates": data["templates"] or []

        }

# user this information for testing porpose 
user_id_testing = "f5f7dea2-d2f9-431c-8529-aea5cd0fa49a"
user_mail = "nofackai@gmail.com"

In [79]:
# Testing
user_mail = "nofackai@gmail.com"

response = get_user_context.invoke({
    "email": user_mail
})


print("User Name:", response["user"]["name"])
print("User Role:", response["user"]["role"])

print("\nFull Context:")
print(response)

User Name: Moksh Bhardwaj 
User Role: AIML Engineer, Gen AI

Full Context:
{'user': {'id': 'f5f7dea2-d2f9-431c-8529-aea5cd0fa49a', 'name': 'Moksh Bhardwaj ', 'email': 'nofackai@gmail.com', 'phone': None, 'role': 'AIML Engineer, Gen AI'}, 'resume': {'raw_text': None, 'skills': None, 'projects': None, 'experience': None, 'education': None, 'certifications': None, 'objective': None, 'social_links': None}, 'assets': [{'name': 'https://mokshbhardwaj.netlify.app/', 'type': 'link', 'description': None, 'content': "Title: Moksh Bhardwaj | Generative AI & Full-Stack AI Engineer\n\nURL Source: https://mokshbhardwaj.netlify.app/\n\nMarkdown Content:\n[![Image 1: Moksh Logo](https://mokshbhardwaj.netlify.app/assets/image-Bve3jEIi.jpg)](https://mokshbhardwaj.netlify.app/)\n*   [Home](https://mokshbhardwaj.netlify.app/)\n*   [About](https://mokshbhardwaj.netlify.app/)\n*   [Skills](https://mokshbhardwaj.netlify.app/)\n*   [Project](https://mokshbhardwaj.netlify.app/)\n*   [Contact](https://mokshbhar

In [80]:
import os
from dotenv import load_dotenv
import requests
from pinecone import Pinecone

# 1. Load API Keys from environment variables
load_dotenv(override=True)
cohere_api_key = os.getenv("COHERE_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX", "outreachx")

# Provide the user_id if you want to search within a specific user's uploaded assets
# Your system saves data under the namespace format: "user_{user_id}"
user_id = user_id_testing

# 2. Initialize Pinecone Client
pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index(pinecone_index_name)

# 3. Define the user query
user_query = "skills"

# 4. Generate the embedding for the query using the Cohere API 
# (Matches the logic in your VectorService)
print("Generating embedding for query...")
url = "https://api.cohere.com/v2/embed"
payload = {
    "model": "embed-v4.0",
    "input_type": "search_query", # For queries, Cohere uses 'search_query'
    "texts": [user_query],
    "output_dimension": 1024,
    "embedding_types": ["float"]
}
headers = {
    "Authorization": f"Bearer {cohere_api_key}",
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)
response.raise_for_status()

# Extract the float embedding list
query_embedding = response.json().get("embeddings", {}).get("float")[0]

# 5. Query the Pinecone Vector Database
print("Querying Pinecone database...")
namespace = f"user_{user_id}"

search_results = index.query(
    vector=query_embedding,
    top_k=5,                 # Retrieve the top 5 closest matches
    include_metadata=True,   # Include the actual text and metadata stored
    namespace=namespace      # Ensure we only search this user's assets
)

# 6. Display the results
print(f"\n--- Top Results for: '{user_query}' ---\n")
for i, match in enumerate(search_results['matches']):
    score = match['score']
    
    # Extract metadata fields saved during upsert
    text_content = match['metadata'].get('text', 'No text found')
    asset_name = match['metadata'].get('name', 'Unknown source')
    asset_type = match['metadata'].get('type', 'Unknown type')
    
    print(f"Result #{i+1} | Match Score: {score:.4f}")
    print(f"Asset Name: {asset_name} ({asset_type})")
    print(f"Snippet: {text_content[:300]}...") # Print the first 300 characters
    print("-" * 60)


Generating embedding for query...
Querying Pinecone database...

--- Top Results for: 'skills' ---

Result #1 | Match Score: 0.0458
Asset Name: Unknown source (technical_doc)
Snippet: Pinecone is a fully managed vector database that makes it easy to add vector search to production apps....
------------------------------------------------------------
Result #2 | Match Score: 0.0458
Asset Name: Unknown source (technical_doc)
Snippet: Pinecone is a fully managed vector database that makes it easy to add vector search to production apps....
------------------------------------------------------------
Result #3 | Match Score: 0.0458
Asset Name: Unknown source (technical_doc)
Snippet: Pinecone is a fully managed vector database that makes it easy to add vector search to production apps....
------------------------------------------------------------
Result #4 | Match Score: 0.0458
Asset Name: Unknown source (technical_doc)
Snippet: Pinecone is a fully managed vector database that makes it ea

In [81]:
import os
import requests
import uuid
from pinecone import Pinecone

# 1. Load API Keys from environment variables
cohere_api_key = os.getenv("COHERE_API_KEY") # Replace if using a different provider like OpenAI
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX_NAME")

# The exact namespace from your error message
namespace = "user_f5f7dea2-d2f9-431c-8529-aea5cd0fa49a"

# 2. Initialize Pinecone Client
pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index(pinecone_index_name)

# 3. Sample Documents (Replace this with your actual file data)
documents = [
    {
        "id": "doc1",
        "text": "OutreachX is an AI platform that personalizes cold emails for high conversion rates.",
        "type": "product_info"
    },
    {
        "id": "doc2",
        "text": "Pinecone is a fully managed vector database that makes it easy to add vector search to production apps.",
        "type": "technical_doc"
    }
]

print("Generating embeddings and storing in Pinecone...")
vectors_to_upsert = []

# 4. Generate Embeddings & Prepare Data
for doc in documents:
    # --- Generate Embedding ---
    # Using Cohere embed-english-v3.0 as it natively outputs 1024 dimensions
    url = "https://api.cohere.com/v2/embed"
    payload = {
        "model": "embed-english-v3.0", 
        "input_type": "search_document", # Use 'search_document' for storing, and 'search_query' for retrieving
        "texts": [doc["text"]],
        "embedding_types": ["float"]
    }
    headers = {
        "Authorization": f"Bearer {cohere_api_key}",
        "Content-Type": "application/json"
    }
    
    response = requests.post(url, json=payload, headers=headers)
    response.raise_for_status()
    
    # Extract the embedding
    embedding = response.json().get("embeddings", {}).get("float")[0]
    
    # --- Prepare Vector Tuple for Upsert ---
    # Pinecone requires tuples of: (id, vector_values, metadata_dict)
    vector_id = str(uuid.uuid4())
    
    metadata = {
        "text": doc["text"],
        "source_id": doc["id"],
        "type": doc["type"]
    }
    
    vectors_to_upsert.append((vector_id, embedding, metadata))

# 5. Upsert Vectors to Pinecone Index
if vectors_to_upsert:
    # Upsert the batch of vectors
    index.upsert(
        vectors=vectors_to_upsert,
        namespace=namespace
    )
    print(f"✅ Successfully upserted {len(vectors_to_upsert)} vectors to namespace '{namespace}'.")
else:
    print("No vectors to upsert.")

# 6. Verify the Stats Again!
print("\n--- Updated Pinecone Stats ---")
print(index.describe_index_stats())


Generating embeddings and storing in Pinecone...
✅ Successfully upserted 2 vectors to namespace 'user_f5f7dea2-d2f9-431c-8529-aea5cd0fa49a'.

--- Updated Pinecone Stats ---
DescribeIndexStatsResponse(dimension=1024, total_vector_count=14, metric='cosine', namespaces=1)


In [82]:
# 5. Template Generation LLM Node
from pydantic import BaseModel, Field

class TemplateOutput(BaseModel):
    subject_line: str = Field(description="The subject line for the cold email")
    text_content: str = Field(description="The plain text version of the cold email. MUST include {{company_name}}.")
    html_content: str = Field(description="The HTML version of the cold email. MUST include {{company_name}}. ONLY generate if explicitly requested, otherwise leave empty.", default="")

def template_generator(state: TemplateAgentState):
    print("-> Generating Final Template...")
    
    query = state["user_query"]
    basic_info = state.get("basic_info", {})
    memory = state.get("memory", [])
    knowledge = state.get("retrieved_knowledge", [])
    
    system_prompt = """
    You are an expert Cold Email and Outreach Template Generator.
    Your task is to draft a complete, highly personalized outreach email for the user based on their profile, their memory, and the retrieved knowledge of their projects and skills.
    
    CRITICAL INSTRUCTIONS - AVOID PLACEHOLDERS:
    - You MUST write the actual email using the user's REAL data (Name, Role, Skills, Projects) provided in the context below. 
    - DO NOT use generic brackets or placeholders like [Insert Project Name], {{skills}}, {{sender_name}}, or {{recipient_name}}. Write it as if it's a finished, polished email ready to be sent!
    - The ONLY placeholder you are allowed to use is {{company_name}}, because the user will swap that out later depending on who they email.
    
    CRITICAL INSTRUCTIONS - FORMATTING:
    1. You MUST generate a 'text_content' version of the template.
    2. Incorporate specific facts and metrics from the user's retrieved knowledge (e.g., specific projects they built, tech stack they use) into the email body so it sounds authentic.
    3. IF AND ONLY IF the user's query or memory explicitly demands HTML code, then populate the 'html_content' field. Otherwise, leave it as an empty string.
    """
    
    user_prompt = f"""
    User Query: {query}
    
    User's Actual Profile (SENDER):
    Name: {basic_info.get('name', 'User')}
    Role: {basic_info.get('role', 'Professional')}
    
    User's Conversation Memory (Preferences/Feedback):
    {memory}
    
    User's Retrieved Knowledge (Actual Skills/Projects/Assets to mention):
    {knowledge}
    
    Write the personalized template now. DO NOT use generic placeholders (except {{company_name}}). Use the real facts provided above.
    """
    
    # Use structured output to guarantee JSON format
    structured_model = advance_model.with_structured_output(TemplateOutput)
    
    try:
        response = structured_model.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ])
        
        # Format the final template nicely for the user to see
        final_str = f"SUBJECT: {response.subject_line}\n\n"
        final_str += "=== PLAIN TEXT VERSION ===\n"
        final_str += f"{response.text_content}\n\n"
        if response.html_content:
            final_str += "=== HTML VERSION ===\n"
            final_str += f"{response.html_content}\n"
            
        return {"final_template": final_str}
        
    except Exception as e:
        print(f"Error generating template: {e}")
        return {"final_template": "Error generating template."}

# Compile the Graph
workflow = StateGraph(TemplateAgentState)

workflow.add_node("aggregator", context_aggregator)
workflow.add_node("planner", query_planner)
workflow.add_node("retriever", vector_retriever)
workflow.add_node("generator", template_generator)

workflow.add_edge(START, "aggregator")
workflow.add_edge("aggregator", "planner")
workflow.add_edge("planner", "retriever")
workflow.add_edge("retriever", "generator")
workflow.add_edge("generator", END)

template_agent = workflow.compile()
print("Template Agent Workflow Compiled Successfully!")

Template Agent Workflow Compiled Successfully!


In [83]:
# Testing the Template Agent Architecture
test_state = {
  "user_query": "Create a professional cold email template for a Generative AI Developer role. HTML code output is mandatory. The final response must contain a complete, production-ready HTML email template with proper structure, styling, responsive design, and placeholders for dynamic variables. Include variables such as {{recipient_name}}, {{recipient_company}}, {{recipient_role}}, {{sender_name}}, {{sender_email}}, {{sender_phone}}, {{sender_linkedin}}, {{sender_portfolio}}, {{sender_github}}, {{company_name}}, {{project_name}}, {{skills}}, {{experience}}, {{achievements}}, {{call_to_action}}, and {{custom_message}}. The email should include a professional subject line, greeting, introduction, value proposition, technical skills section, project/experience highlights, closing message, and signature. Ensure the HTML is clean, reusable, and suitable for sending through an email platform.",
  "user_id": user_id_testing,
  "email": user_mail
}

print("Running Template Agent...")
final_state = template_agent.invoke(test_state)

print("\n" + "="*50)
print("FINAL GENERATED TEMPLATE:")
print("="*50)
print(final_state["final_template"])


Running Template Agent...
-> Fetching Context from DB...
-> Planning Vector DB Search Query...
Generated Search Query: `((skill:("Python" OR "Java" OR "C++") AND (project:"Generative AI" OR experience:"AI Development")) OR (achievement:(" Published research paper" OR "Developed AI model") AND role:"AIML Engineer" OR company:"Gen AI"))`
-> Retrieving Knowledge from Pinecone...
-> Generating Final Template...

FINAL GENERATED TEMPLATE:
SUBJECT: Generative AI Expertise for {{company_name}} - Moksh Bhardwaj, AIML Engineer

=== PLAIN TEXT VERSION ===
Dear {{recipient_name}},

My name is Moksh Bhardwaj, and I am an AIML Engineer specializing in Generative AI. I'm reaching out because I'm deeply impressed by the innovative work {{company_name}} is doing in the AI space, and I see a strong alignment with my expertise.

As a Generative AI Developer, I possess a robust background in developing and deploying advanced AI/ML solutions. I am passionate about building cutting-edge generative models a